# Model training for building permit cost prediction

We train four regression models -- Ridge, Random Forest, Gradient Boosting, and
XGBoost -- to predict `log_cost`. The training pipeline uses the helper functions
in `src/model.py`, which handle label encoding, scaling, fitting, and metric
computation. We add cross-validation and learning-curve analysis on top.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, learning_curve

from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features
from src.model import (
    prepare_model_data, train_models, get_feature_importance,
    save_model, TARGET,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load the engineered dataset

If the engineered CSV from notebook 02 is available we load it directly;
otherwise we re-run the full pipeline.

In [ ]:
import os

engineered_path = '../data/building_permits_engineered.csv'
if os.path.exists(engineered_path):
    df = pd.read_csv(engineered_path, low_memory=False)
    print(f'Loaded engineered data: {len(df):,} rows')
else:
    raw = load_or_fetch_data('../data', limit=500000)
    df = preprocess_data(raw)
    df = engineer_features(df)
    print(f'Built engineered data on the fly: {len(df):,} rows')

## 2. Prepare feature matrix and target vector

`prepare_model_data` from `model.py` applies label encoding to categorical
features, selects available numeric and categorical columns, fills remaining
NaNs with column medians, and returns `(X, y, label_encoders, feature_names)`.

In [ ]:
X, y, label_encoders, feature_names = prepare_model_data(df)
print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Features used: {feature_names}')

## 3. Train all four models with an 80/20 split

`train_models` performs the split, scales features for Ridge, trains each
model, and computes MAE, RMSE, R-squared, and MAPE on the hold-out set.
Predictions are converted back from log space for interpretable error metrics.

In [ ]:
trained_models, results, scaler, X_test, y_test = train_models(X, y, random_state=42)

results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
results_df = results_df.round(4)
print(results_df.to_string())

## 4. Cross-validation

A single train/test split can be sensitive to the random seed. Five-fold
cross-validation gives a more robust estimate of generalisation performance.
We use negative mean absolute error as the scoring metric (sklearn convention).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

cv_models = {
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0)),
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_split=10,
        random_state=42, n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42,
    ),
    'XGBoost': XGBRegressor(
        n_estimators=300, max_depth=7, learning_rate=0.1,
        random_state=42, n_jobs=-1,
    ),
}

cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    cv_results[name] = {'CV MAE mean': -scores.mean(), 'CV MAE std': scores.std()}
    print(f'{name:25s}  MAE = {-scores.mean():.4f} +/- {scores.std():.4f}')

cv_df = pd.DataFrame(cv_results).T.round(4)
cv_df

## 5. Learning curves

Learning curves show how training and validation error change as we increase
the training set size. A large gap between the two curves signals overfitting;
both curves plateauing at a high error signals underfitting.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, model) in zip(axes.flat, cv_models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
    )
    train_mean = -train_scores.mean(axis=1)
    val_mean = -val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'o-', label='Training MAE')
    ax.plot(train_sizes, val_mean, 's-', label='Validation MAE')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1)
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Training set size')
    ax.set_ylabel('MAE (log-cost)')
    ax.legend(fontsize=9)

plt.suptitle('Learning curves for all four models', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Model comparison summary

We combine the hold-out metrics from `train_models` with the cross-validation
results into a single table for easy comparison.

In [ ]:
summary = results_df.copy()
summary['CV MAE mean'] = cv_df['CV MAE mean']
summary['CV MAE std'] = cv_df['CV MAE std']
summary = summary.sort_values('R2', ascending=False)

print('Model comparison (sorted by R-squared):\n')
print(summary.to_string())

In [ ]:
# Bar chart of R-squared values
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if v == summary['R2'].max() else '#95a5a6' for v in summary['R2']]
summary['R2'].plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.set_xlabel('R-squared')
ax.set_title('Model comparison by R-squared', fontsize=12)
ax.axvline(x=0.85, ls='--', color='red', alpha=0.5, label='Baseline target (0.85)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Select and save the best model

XGBoost typically achieves the highest R-squared on this dataset. We persist it
along with the scaler, label encoders, and feature list so that `04_evaluation`
and any deployment pipeline can reload them.

In [ ]:
best_name = summary['R2'].idxmax()
best_model = trained_models[best_name]
print(f'Best model: {best_name} (R2 = {summary.loc[best_name, "R2"]:.4f})')

save_model(best_model, scaler, label_encoders, feature_names, model_dir='../models')
print('Model artifacts saved to ../models/')

## Summary

- Trained Ridge, Random Forest, Gradient Boosting, and XGBoost on ~484K permits.
- Five-fold cross-validation confirmed that XGBoost has the lowest MAE variance.
- Learning curves show tree-based models benefit from additional data while Ridge
  plateaus early, suggesting the relationship is non-linear.
- XGBoost was selected as the best model and saved for evaluation in notebook 04.